In [1]:
from pathlib import Path
import polars as pl

### Load files

In [2]:
bronze_dir = Path('../data/bronze/buildings')
silver_dir = Path('../data/silver/buildings')

bronze_files = sorted(bronze_dir.glob('*.parquet'))
silver_files = sorted(silver_dir.glob('*.parquet'))

### Silver file count check
1. Collect bronze & silver file names in a set
2. Compare bronze & silver counts
3. Check what are the missing/unexpected files in silver

In [3]:
bronze_names = {file.name for file in bronze_files}
silver_names = {file.name for file in silver_files}

if len(silver_names) == len(bronze_names):
    print('File count OK')
else:
    print(f'File count mismatch: bronze={len(bronze_names)}, silver={len(silver_names)}')

missing_in_silver = sorted(bronze_names - silver_names)
unexpected_in_silver = sorted(silver_names - bronze_names)

if not missing_in_silver and not unexpected_in_silver:
    print('Bronze and Silver filename mapping OK')
else:
    if missing_in_silver:
        print('Missing in silver:')
        for name in missing_in_silver:
            print(name)
    
    if unexpected_in_silver:
        print('Unexpected in silver:')
        for name in unexpected_in_silver:
            print(name)

File count OK
Bronze and Silver filename mapping OK


### Column and geometry checks
1. Set expected column names in a set
2. Loop through silver files
3. Check if each file's column names are as expected
4. Check for null geometries and empty files

In [4]:
all_ok = True
expected_columns = {
    'country',
    'quadkey',
    'upload_date',
    'source_file',
    'source_row_index',
    'silver_record_id',
    'geometry',
    'geometry_type',
}

for file in silver_files:
    schema_names = set(pl.scan_parquet(file).collect_schema().names())

    if 'geometry' not in schema_names:
        print(f'Missing geometry in {file.name}')
        all_ok = False
        continue

    if 'geometry_type' not in schema_names:
        print(f'Missing geometry_type in {file.name}')
        all_ok = False
        continue

    unexpected_columns = sorted(schema_names - expected_columns)
    if unexpected_columns:
        print(f'{file.name}: unexpected columns {unexpected_columns}')
        all_ok = False

    result = (
        pl.scan_parquet(file)
        .select(
            pl.len().alias('rows'),
            pl.col('geometry').null_count().alias('geometry_nulls'),
            pl.col('geometry_type').null_count().alias('geometry_type_nulls'),
        )
        .collect()
        .row(0, named=True)
    )

    if result['rows'] == 0:
        print(f'Empty silver file: {file.name}')
        all_ok = False
        continue

    if result['geometry_nulls'] > 0:
        print(f"{file.name}: geometry has {result['geometry_nulls']} null rows")
        all_ok = False

    if result['geometry_type_nulls'] > 0:
        print(f"{file.name}: geometry_type has {result['geometry_type_nulls']} null rows")
        all_ok = False

if all_ok:
    print('Silver geometry columns OK')

Silver geometry columns OK
